# Q11 — Does BPMA remain useful under family misspecification?

Hypothesis: BPMA should select or average the best available approximations when the true family is absent, but family shares should not be interpreted as proof that a family is scientifically true. Add distractor families and monitor efficiency.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
while root != root.parent and not (root / 'bayesian_predictive_model_averaging').is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from sklearn.metrics import log_loss
from bayesian_predictive_model_averaging import default_family_registry
from EXPERIMENTS.common import classification_data, split_data, fit_classifier

In [2]:
X, y = classification_data(seed=41, kind='moons')
X_train, X_test, y_train, y_test = split_data(X, y, seed=41)
registry = default_family_registry()
registries = {
    'full registry': registry,
    'linear-only': tuple(r for r in registry if r.adapter.name == 'linear_mixture'),
    'nonlinear families': tuple(r for r in registry if r.adapter.name != 'linear_mixture'),
}
results = {}
for label, candidate_registry in registries.items():
    model = fit_classifier(X_train, y_train, seed=41, family_registry=candidate_registry)
    results[label] = {
        'test_log_loss': log_loss(y_test, model.predict_proba(X_test)),
        'family_mass': model.get_model_masses()['family'],
        'ess_fraction': model.effective_sample_size_fraction_,
    }
results

{'full registry': {'test_log_loss': 0.23453453355127674,
  'family_mass': {'knn': 0.5020655679031382,
   'linear_mixture': 0.12275781626067435,
   'mlp': 0.12404866595564576,
   'random_forest': 0.2511279498805417},
  'ess_fraction': 0.999000544757581},
 'linear-only': {'test_log_loss': 0.3144320110671534,
  'family_mass': {'linear_mixture': 1.0},
  'ess_fraction': 0.9994967147346504},
 'nonlinear families': {'test_log_loss': 0.22531710479453324,
  'family_mass': {'gaussian_mixture': 0.12570278047005282,
   'knn': 0.5003800981940397,
   'mlp': 0.12363222578868605,
   'random_forest': 0.25028489554722144},
  'ess_fraction': 0.9990482983025127}}

Add weak and strong distractors, duplicated families, and an over-flexible family with a controlled prior. Repeat across generators and seeds. Interpret family mass as predictive contribution, not as evidence of a true data-generating mechanism.

## Conclusion from the executed starter run

**Status: not falsified; preliminary support.** On the moons benchmark, linear-only BPMA had log loss 0.3144, versus 0.2345 for the full registry and 0.2253 for the nonlinear-only registry. Removing an unsuitable family improved this result, consistent with sensitivity to family misspecification. The run does not yet include a known generating family or controlled distractors.